In [24]:
from typing import Dict, Any

In [49]:
def setup(q: int, m: int, n: int, w:int)->Dict[str,Any]:
    F = GF(q)
    Fext.<z> = F.extension(m)
    Tmp.<y> = PolynomialRing(F)
    Q = Tmp.random_element(degree=n)
    while not Q.is_irreducible():
        Q = Tmp.random_element(degree=n)
    
    PR.<x> = PolynomialRing(Fext)
    Q = PR(Q)

    return {
        "Q": Q,
        "F": F,
        "Fext": Fext,
        "q": q,
        "m": m,
        "n": n,
        "w": w,
        "x": x
    }

In [47]:
def vec(a):
    return a.matrix().column(0)

In [50]:
def mat(a, params):
    F, m = params["F"], params["m"]
    res = zero_matrix(F, m, len(a))
    for i, elem in enumerate(a):
        res[:, i] = vec(elem)

    return res

In [52]:
def rank_norm(a, params):
    return rank(mat(a, params))

In [83]:
def find_B(f, params):
    F, m = params["F"], params["m"]
    B = mat(f, params)
    cand_cols = identity_matrix(F,m)
    for i in range(m):
        tmp = B.augment(cand_cols[:,i])
        if rank(tmp) == rank(B) + 1:
            B = tmp
        if B.ncols() == m:
            break

    return B

In [84]:
def sample_with_supp(f, params):
    F = params["F"]
    c = random_vector(F, len(f))

    return f.inner_product(c)

In [85]:
def poly(v, params):
    x = params["x"]

    return sum([v[i]*x**i for i in range(len(v))])

In [86]:
def poly_to_vec(p,d, params):
    Fext = params["Fext"]

    return vector(Fext,[p.coefficient(i) for i in range(d+1)])

In [108]:
def vec_vec_prod(u, v, params):
    Q = params["Q"]

    u_poly = poly(u, params)
    v_poly = poly(v, params)
    tmp = (u_poly * v_poly) % Q

    return poly_to_vec(tmp, len(u)-1, params)

In [109]:
def get_f(params):
    Fext, w = params["Fext"], params["w"]
    
    f = random_vector(Fext,w)
    while rank_norm(f, params) != w:
        f = random_vector(Fext,w)

    return f

In [110]:
def get_g(B,params):
    Fext, m, w = params["Fext"], params["m"], params["w"]
    z = Fext.gen() 

    z_pows = vector(Fext,[z**i for i in range(m)])
    g = z_pows * B[:,w:]

    return g

In [111]:
def key_gen(params: Dict[str,Any]):
    f = get_f(params)
    B = find_B(f, params)
    g = get_g(B, params)
    D = (B**(-1)).T[:,params["w"]:]
    s = vector(params["Fext"], [sample_with_supp(f, params) for _ in range(params["n"])])

    return f,g,D,s

In [112]:
def encrypt(message, key, params):
    f,g,D,s = key
    Fext, n, F, w = params["Fext"], params["n"], params["F"], params["w"]
    
    u = random_vector(Fext,n)
    p = poly(u,params)
    R2 = random_matrix(F,w,n)
    e = f*R2
    v = vec_vec_prod(s,u, params) + e + (message*g[0])
    ct = (u,v)

    return ct

In [113]:
def decrypt(ct, key, params):
    f,g,D,s = key
    
    u,v = ct
    d = D[:,0]

    message = (d).T * mat(v - vec_vec_prod(s, u, params), params)

    return message

In [120]:
params = setup(2, 15, 11, 3)
key = key_gen(params)

F, n = params["F"], params["n"]
m1 = random_vector(F, n)
m2 = random_vector(F, n)

ct1 = encrypt(m1, key, params)
ct2 = encrypt(m2, key, params)

assert vector(decrypt(ct1, key, params)[0]) == m1
print("Encrypt/Decrypt")

ct_sum = (ct1[0]+ct2[0], ct1[1]+ct2[1])
assert vector(decrypt(ct_sum, key, params)[0]) == m1 + m2
print("Сложение")

Encrypt/Decrypt
Сложение


In [208]:
def get_g_she(B,d_F,params):
    Fext, m = params["Fext"], params["m"]
    z = Fext.gen() 

    z_pows = vector(Fext,[z**i for i in range(m)])
    g = z_pows * B[:,d_F:]

    return g

In [202]:
def key_gen_she(params: Dict[str,Any]):
    Fext = params["Fext"]
    w = params["w"]
        
    f = get_f(params)
    
    rw = 0
    d_F = 1
    while d_F + 2!= rw:
        g1 = Fext.random_element()
        fg1 = f*g1
    
        gens = (
            list(f) +
            list(fg1) +
            [f[i] * f[j] for i in range(w) for j in range(i,w)]
        )
        
        M = matrix(params["F"], [vec(g) for g in gens])
    
        F_ = M.row_space().basis()
        g2 = g1*g1
    
        f_ = vector(Fext, [Fext(list(b)) for b in F_])
        check = vector(Fext, list(f_) + [g1, g2])

        d_F = len(f_)
        rw = rank_norm(check,params)

    B = find_B(check, params)
    g = get_g_she(B,d_F, params)
    D = (B**(-1)).T[:,d_F:]
    s = vector(params["Fext"], [sample_with_supp(f, params) for _ in range(params["n"])])

    return f,g,D,s

In [203]:
def mul_ct(ct1,ct2, params):
    u1,v1 = ct1
    u2,v2 = ct2

    a = vec_vec_prod(v1,v2,params)
    b = -(vec_vec_prod(u1,v2,params) + vec_vec_prod(u2,v1,params))
    c = vec_vec_prod(u1,u2,params)
    
    return a,b,c

In [204]:
def decrypt_mul(ct, key, params):
    f,g,D,s = key
    a,b,c = ct

    sb = vec_vec_prod(s,b,params)
    ss = vec_vec_prod(s,s,params)
    s2c = vec_vec_prod(ss,c,params)

    tmp = a + sb + s2c

    d = D[:,1]

    message = (d).T * mat(tmp, params)

    return message

In [270]:
class AHE:
    def __init__(self, q: int, m: int, n: int, w:int):
        self.q = q
        self.m = m
        self.n = n
        self.w = w
        
        self.F = GF(q)
        self.Fext = GF(q^m, 'z')
        
        Tmp.<y> = PolynomialRing(self.F)
        Q = Tmp.random_element(degree=self.n)
        while not Q.is_irreducible():
            Q = Tmp.random_element(degree=self.n)
        
        self.PR = PolynomialRing(self.Fext, 'x')
        self.x = self.PR.gen()
        self.Q = self.PR(Q)

    @staticmethod
    def vec(a):
        return a.matrix().column(0)

    def _mat(self, a):
        res = zero_matrix(self.F, self.m, len(a))
        for i, elem in enumerate(a):
            res[:, i] = self.vec(elem)

        return res

    def _rank_norm(self, a):
        return rank(self._mat(a))

    def _find_B(self, f):
        B = self._mat(f)
        cand_cols = identity_matrix(self.F, self.m)
        for i in range(self.m):
            tmp = B.augment(cand_cols[:,i])
            if rank(tmp) == rank(B) + 1:
                B = tmp
            if B.ncols() == self.m:
                break
    
        return B

    def _sample_with_supp(self, f):
        c = random_vector(self.F, len(f))
    
        return f.inner_product(c)

    def _poly(self, v):
        return sum([v[i]*self.x**i for i in range(len(v))])

    def _poly_to_vec(self, p, d):
        return vector(self.Fext,[p.coefficient(i) for i in range(d+1)])

    def _vec_vec_prod(self, u, v):
        u_poly = self._poly(u)
        v_poly = self._poly(v)
        tmp = (u_poly * v_poly) % self.Q
    
        return self._poly_to_vec(tmp, len(u)-1)

    def _get_f(self): 
        f = random_vector(self.Fext, self.w)
        while self._rank_norm(f) != self.w:
            f = random_vector(self.Fext,self.w)
    
        return f

    def _get_g(self, B):
        z = self.Fext.gen() 
    
        z_pows = vector(self.Fext,[z**i for i in range(self.m)])
        g = z_pows * B[:,self.w:]

        return g

    def key_gen(self):
        f = self._get_f()
        B = self._find_B(f)
        g = self._get_g(B)
        D = (B**(-1)).T[:,self.w:]
        s = vector(self.Fext, [self._sample_with_supp(f) for _ in range(self.n)])

        return f,g,D,s

    def encrypt(self, message, key):
        f,g,D,s = key
        
        u = random_vector(self.Fext, self.n)
        p = self._poly(u)
        R2 = random_matrix(self.F,self.w,self.n)
        e = f*R2
        v = self._vec_vec_prod(s,u) + e + (message*g[0])
        ct = (u,v)
    
        return ct

    def decrypt(self, ct, key):
        f,g,D,s = key
        
        u,v = ct
        d = D[:,0]
    
        message = (d).T * self._mat(v - self._vec_vec_prod(s, u))
    
        return vector(message)


    @staticmethod
    def sum(ct1, ct2):
        return (ct1[0] + ct2[0], ct1[1] + ct2[1])

    def mul_const(self, ct, msg):
        u = self._vec_vec_prod(ct[0], msg, self.Q)
        v = self._vec_vec_prod(ct[1], msg, self.Q)

        return (u, v)

In [284]:
ahe = AHE(2, 15, 11, 3)
sk = ahe.key_gen()

msg1 = random_vector(ahe.F, ahe.n)
msg2 = random_vector(ahe.F, ahe.n)

ct1 = ahe.encrypt(msg1, sk)
ct2 = ahe.encrypt(msg2, sk)

ct3 = ahe.sum(ct1,ct2)

print(ahe.decrypt(ct3,sk) == msg1+msg2)

True


In [280]:
class SHE(AHE):
    def _get_g(self, B, d_F):
        z = self.Fext.gen() 
    
        z_pows = vector(self.Fext,[z**i for i in range(self.m)])
        g = z_pows * B[:,d_F:]
    
        return g

    def key_gen(self):
        f = self._get_f()
        
        rw = 0
        d_F = 1
        while d_F + 2!= rw:
            g1 = self.Fext.random_element()
            fg1 = f*g1
        
            gens = (
                list(f) +
                list(fg1) +
                [f[i] * f[j] for i in range(self.w) for j in range(i, self.w)]
            )
            
            M = matrix(self.F, [self.vec(g) for g in gens])
        
            F_ = M.row_space().basis()
            g2 = g1*g1
        
            f_ = vector(self.Fext, [self.Fext(list(b)) for b in F_])
            check = vector(self.Fext, list(f_) + [g1, g2])
    
            d_F = len(f_)
            rw = self._rank_norm(check)
    
        B = self._find_B(check)
        g = self._get_g(B,d_F)
        D = (B**(-1)).T[:,d_F:]
        s = vector(self.Fext, [self._sample_with_supp(f) for _ in range(self.n)])
    
        return f,g,D,s

    def mul_ct(self, ct1, ct2):
        u1,v1 = ct1
        u2,v2 = ct2
    
        a = self._vec_vec_prod(v1,v2)
        b = -(self._vec_vec_prod(u1,v2) + self._vec_vec_prod(u2,v1))
        c = self._vec_vec_prod(u1,u2)
    
        return a,b,c

    def decrypt_mul(self, ct, key):
        f,g,D,s = key
        a,b,c = ct
    
        sb = self._vec_vec_prod(s,b)
        ss = self._vec_vec_prod(s,s)
        s2c = self._vec_vec_prod(ss,c)
    
        tmp = a + sb + s2c
    
        d = D[:,1]
    
        message = (d).T * self._mat(tmp)
    
        return vector(message)

In [283]:
she = SHE(2, 15, 11, 3)
sk = she.key_gen()

m1 = random_vector(she.F, she.n)
m2 = random_vector(she.F, she.n)

ct1 = she.encrypt(m1, sk)
ct2 = she.encrypt(m2, sk)

ct_mul = she.mul_ct(ct1, ct2)
result = she.decrypt_mul(ct_mul, sk)
expected = she._vec_vec_prod(m1, m2)

print(result == expected)

True
